In [0]:
# ---------------------------------------------------------------------------
# 1. Configuration & Secret Retrieval (never log or print secret values)
# ---------------------------------------------------------------------------
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

SECRET_SCOPE = "kv-aircargo-scope"
LISTEN_CONN_SECRET_KEY = "eh-flight-telemetry-listen-conn"
EVENT_HUB_NAME = "flight-telemetry"

CATALOG_NAME = "aircargo_dev"
BRONZE_SCHEMA = "bronze"
BRONZE_TABLE = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.flight_telemetry_raw"
QUARANTINE_TABLE = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.flight_telemetry_quarantine"

# Checkpoint paths stored under ADLS Gen2 / Unity Catalog volume or external location
CHECKPOINT_BASE = "/Volumes/aircargo_dev/landing/raw_files/_checkpoints/bronze_flight_telemetry"
MAIN_CHECKPOINT_PATH = f"{CHECKPOINT_BASE}/flight_telemetry_raw"
QUARANTINE_CHECKPOINT_PATH = f"{CHECKPOINT_BASE}/flight_telemetry_quarantine"

raw_listen_conn = dbutils.secrets.get(scope=SECRET_SCOPE, key=LISTEN_CONN_SECRET_KEY)
assert raw_listen_conn and len(raw_listen_conn.strip()) > 0, "Empty Event Hubs listen connection string"

# Ensure EntityPath is present in the connection string for the Spark Event Hubs connector
if "EntityPath=" not in raw_listen_conn:
    eh_connection_string = f"{raw_listen_conn.rstrip(';')};EntityPath={EVENT_HUB_NAME}"
else:
    eh_connection_string = raw_listen_conn

In [0]:
# ---------------------------------------------------------------------------
# 2. Target Bronze & Quarantine Delta Tables Setup
# ---------------------------------------------------------------------------
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    event_id STRING,
    payload_json STRING,
    event_hub_name STRING,
    partition_id STRING,
    event_offset STRING,
    enqueued_time TIMESTAMP,
    _ingest_timestamp TIMESTAMP,
    _source_system STRING
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} (
    raw_body_text STRING,
    quarantine_reason STRING,
    event_hub_name STRING,
    partition_id STRING,
    event_offset STRING,
    enqueued_time TIMESTAMP,
    _ingest_timestamp TIMESTAMP,
    _source_system STRING
)
USING DELTA
""")

In [0]:
# ---------------------------------------------------------------------------
# 3. Read Stream from Event Hubs (Kafka Protocol) & Split Valid vs. Quarantine
# ---------------------------------------------------------------------------
EVENT_HUBS_NAMESPACE = "evhns-aircargo-dev-jm.servicebus.windows.net"

eh_kafka_options = {
    "kafka.bootstrap.servers": f"{EVENT_HUBS_NAMESPACE}:9093",
    "subscribe": EVENT_HUB_NAME,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{eh_connection_string}";'
    ),
    "startingOffsets": "earliest",
    "maxOffsetsPerTrigger": 1000,
}

raw_stream_df = (
    spark.readStream
    .format("kafka")
    .options(**eh_kafka_options)
    .load()
)

minimal_flight_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("icao24", StringType(), True),
])

parsed_check_df = (
    raw_stream_df
    .withColumn("raw_body_text", F.col("value").cast(StringType()))
    .withColumn("parsed_check", F.from_json(F.col("raw_body_text"), minimal_flight_schema))
    .withColumn("event_hub_name", F.col("topic"))
    .withColumn("partition_id", F.col("partition").cast(StringType()))
    .withColumn("event_offset", F.col("offset").cast(StringType()))
    .withColumn("enqueued_time", F.col("timestamp").cast("timestamp"))
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source_system", F.lit("azure_event_hubs_opensky"))
)

valid_bronze_df = (
    parsed_check_df
    .filter(F.col("raw_body_text").isNotNull() & F.col("parsed_check").isNotNull())
    .select(
        F.col("parsed_check.event_id").alias("event_id"),
        F.col("raw_body_text").alias("payload_json"),
        "event_hub_name",
        "partition_id",
        "event_offset",
        "enqueued_time",
        "_ingest_timestamp",
        "_source_system",
    )
)

quarantine_df = (
    parsed_check_df
    .filter(F.col("raw_body_text").isNull() | F.col("parsed_check").isNull())
    .withColumn(
        "quarantine_reason",
        F.when(F.col("raw_body_text").isNull(), F.lit("NULL_EVENT_BODY"))
         .otherwise(F.lit("MALFORMED_OR_UNEXPECTED_JSON_SCHEMA"))
    )
    .select(
        "raw_body_text",
        "quarantine_reason",
        "event_hub_name",
        "partition_id",
        "event_offset",
        "enqueued_time",
        "_ingest_timestamp",
        "_source_system",
    )
)

In [0]:
# ---------------------------------------------------------------------------
# 4. Write Micro-Batches Using Trigger.AvailableNow
# ---------------------------------------------------------------------------
bronze_query = (
    valid_bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", MAIN_CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)

quarantine_query = (
    quarantine_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", QUARANTINE_CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(QUARANTINE_TABLE)
)

bronze_query.awaitTermination()
quarantine_query.awaitTermination()

print("Bronze and Quarantine AvailableNow micro-batch ingestion completed.")

In [0]:
# ---------------------------------------------------------------------------
# 5. Quick Verification Query
# ---------------------------------------------------------------------------
# display(spark.sql(f"SELECT COUNT(*) AS bronze_row_count FROM {BRONZE_TABLE}"))
# display(spark.sql(f"SELECT COUNT(*) AS quarantine_row_count FROM {QUARANTINE_TABLE}"))
# display(spark.sql(f"SELECT * FROM {BRONZE_TABLE} ORDER BY _ingest_timestamp DESC LIMIT 10"))